This notebook implements the **SWOOP (SWOT Obstruction Profile)**
workflow for detecting potential obstruction-induced step changes in RiverSP observations.

Building on the application-ready datasets generated in the
`01_Preprocess` preprocessing workflow, this notebook develops and evaluates methods for identifying step-like discontinuities in along-reach water surface elevation (WSE) profiles that may indicate the presence of river obstructions such as dams, weirs, or hydraulic controls.

The final outputs include candidate obstructions on both pilot datasets (train&valid) and global (test) datasets, engineered node-level features, and the mannuel labels.
These results support both methodological development and large-scale analysis of river obstruction patterns derived from SWOT observations.

Intermediate preprocessing datasets are not distributed with the repository
due to their large size, but they can be regenerated using the provided
workflow.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score,
    classification_report,
    accuracy_score,
)

# 1 Pilot experiment on the train_valid dataset


This step performs the initial pilot experiment for the SWOT RiverSP
obstruction analysis. The initial goal is to run a small-scale valid on the
`train_valid` dataset and evaluate whether step-like changes can be detected
from along-reach water surface elevation profiles.

For each reach, the algorithm searches for abrupt changes in the WSE
profile that may correspond to hydraulic controls or obstruction
structures. 

The output of this step is a table of candidate step-change locations and their
associated diagnostic metrics, which will be used for subsequent inspection,
filtering, and model development.

In [4]:
PROJECT_ROOT = Path().resolve().parents[1]

# Input/output directories
INPUT_DIR = PROJECT_ROOT / "data" / "input_data"
OUTPUT_DIR = PROJECT_ROOT / "data" / "results"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [5]:
PROJECT_ROOT

WindowsPath('D:/github/SWOT_RiverSP_Obstruction')

In [6]:
def detect_step_change(
    denoised_signal,
    sig_signal,
    occu_signal,
    std_signal,
    width_signal,
    stability_scale=0.3
):
    """
    Detect candidate step changes along a 1D reach profile.

    Parameters
    ----------
    denoised_signal : array-like
        Node-level median WSE signal along a reach.
    sig_signal : array-like
        Node-level mean sigma0 values.
    occu_signal : array-like
        Node-level mean GSW occurrence values.
    std_signal : array-like
        Node-level WSE standard deviation values.
    width_signal : array-like
        Node-level mean width values.
    stability_scale : float, optional
        Multiplier used to define the residual stability threshold.

    Returns
    -------
    tuple
        (results, has_step), where:
        - results is a list of candidate step-change metrics
        - has_step is a boolean flag
    """

    denoised_signal = np.asarray(denoised_signal)
    sig_signal = np.asarray(sig_signal)
    occu_signal = np.asarray(occu_signal)
    std_signal = np.asarray(std_signal)
    width_signal = np.asarray(width_signal)

    differences = np.abs(np.diff(denoised_signal))
    step_points = np.where(differences > 0.5)[0] + 1

    window_size1_default = 10
    window_size2_default = 10

    results = []

    if len(step_points) == 0:
        return None, False

    signal_range = np.max(denoised_signal) - np.min(denoised_signal)
    if signal_range == 0:
        return None, False

    n = len(denoised_signal)

    # Cluster nearby candidate points and keep only the strongest one
    min_gap_nodes = 2
    min_sep = min_gap_nodes + 1

    step_points_sorted = np.sort(step_points)

    clusters = []
    current_cluster = [step_points_sorted[0]]

    for p in step_points_sorted[1:]:
        if p - current_cluster[-1] <= min_sep:
            current_cluster.append(p)
        else:
            clusters.append(current_cluster)
            current_cluster = [p]

    clusters.append(current_cluster)

    filtered_step_points = []
    for cluster in clusters:
        cluster = np.asarray(cluster)
        best_point = cluster[np.argmax(differences[cluster - 1])]
        filtered_step_points.append(int(best_point))

    # valid stability around each filtered candidate point
    for step_index in filtered_step_points:
        w1 = window_size1_default
        w2 = window_size2_default

        if step_index < w1 or step_index >= n - w2:
            w1 = min(w1, step_index)
            w2 = min(w2, n - step_index)

        if w1 <= 4 or w2 <= 5:
            continue

        before_segment = denoised_signal[step_index - w1: step_index]
        after_segment = denoised_signal[step_index + 1: step_index + w2]

        change_size1 = differences[step_index - 1]

        before_x = np.arange(len(before_segment))
        after_x = np.arange(len(after_segment))

        try:
            before_trend_coeff = np.polyfit(before_x, before_segment, 1)
            after_trend_coeff = np.polyfit(after_x, after_segment, 1)
        except Exception:
            continue

        before_fit = np.polyval(before_trend_coeff, before_x)
        after_fit = np.polyval(after_trend_coeff, after_x)

        before_residuals = before_segment - before_fit
        after_residuals = after_segment - after_fit

        before_residual_std = np.std(before_residuals)
        after_residual_std = np.std(after_residuals)

        before_extrap = np.polyval(before_trend_coeff, len(before_segment))
        after_extrap = np.polyval(after_trend_coeff, -1)

        change_size2 = abs(before_extrap - after_extrap)
        theta = np.arctan(before_trend_coeff[0])
        change_size2 = change_size2 * np.cos(theta)
        change_size3 = (before_extrap - after_extrap) * np.cos(theta)

        change_size = min(change_size1, change_size2)
        stability_threshold = stability_scale * change_size

        l = max(0, step_index - 5)
        r = min(n, step_index + 5)

        left_idx = np.arange(l, step_index)
        right_idx = np.arange(step_index + 1, r)
        idx = np.concatenate([left_idx, right_idx])

        if idx.size == 0:
            continue

        # Sigma0 metrics
        sig = np.nanmean(sig_signal[idx])
        sig2 = np.nanmean(sig_signal[left_idx]) if left_idx.size > 0 else np.nan
        sig3 = np.nanmean(sig_signal[right_idx]) if right_idx.size > 0 else np.nan
        sig4 = sig_signal[step_index]

        # GSW occurrence metrics
        occu = np.nanmean(occu_signal[idx])
        occu2 = np.nanmean(occu_signal[left_idx]) if left_idx.size > 0 else np.nan
        occu3 = np.nanmean(occu_signal[right_idx]) if right_idx.size > 0 else np.nan
        occu4 = occu_signal[step_index]

        time_std = np.mean(std_signal[idx]) / signal_range

        width_vals = width_signal[idx]
        width_mean = np.mean(width_vals)
        width_std = np.std(width_vals)

        if (
            before_residual_std < stability_threshold
            and after_residual_std < stability_threshold
        ):
            results.append([
                step_index,
                change_size,
                change_size1,
                change_size2,
                change_size3,
                signal_range,
                sig,
                sig2,
                sig3,
                sig4,
                occu,
                occu2,
                occu3,
                occu4,
                time_std,
                before_residual_std,
                after_residual_std,
                before_trend_coeff[0],
                after_trend_coeff[0],
                width_mean,
                width_std,
            ])

    if len(results) > 0:
        return results, True

    return None, False

In [14]:
# Main train_valid dataset
train_file = INPUT_DIR / "node_train_valid_v5.10.csv"
df = pd.read_csv(train_file)

# Node-level ancillary variables
sigma_file = INPUT_DIR / "node_sigma0_v5.10.csv"
occu_file = INPUT_DIR / "node_occu_mean.csv"

df_sig = pd.read_csv(sigma_file)
df_occu = pd.read_csv(occu_file)

# Replace nodata occurrence values
df_occu.loc[df_occu["occurrence"] == 255, "occurrence"] = 0

# Derive reach_id from node_id
df["reach_id"] = df["node_id"] // 10000 * 10 + df["node_id"] % 10

In [15]:
# Compute node-level median WSE
median_df = (
    df.groupby("node_id")["wse"]
    .median()
    .reset_index(name="wse_median")
)

# Compute node-level WSE standard deviation
std_df = (
    df.groupby("node_id")["wse"]
    .std()
    .reset_index(name="wse_std")
)

# Compute node-level mean width
width_df = (
    df.groupby("node_id")["width"]
    .mean()
    .reset_index(name="width")
)

# Merge node-level summary statistics
average_df = median_df.merge(std_df, on="node_id")
average_df = average_df.merge(width_df, on="node_id")

# Merge ancillary variables
average_df = average_df.merge(df_sig, on="node_id")
average_df = average_df.merge(df_occu, on="node_id")

# Sort nodes and derive reach_id
average_df = average_df.sort_values(by="node_id").reset_index(drop=True)
average_df["reach_id"] = (
    average_df["node_id"] // 10000 * 10 + average_df["node_id"] % 10
)

average_df.head()

,node_id,wse_median,wse_std,width,rdr_sig0,Unnamed: 0,occurrence,reach_id
0,1.141000e+13,1.26915,1.227005,290.097088,45.419136,62,28.40,1.141000e+10
1,1.141000e+13,1.23681,1.670471,259.807184,40.149205,63,26.84,1.141000e+10
2,1.141000e+13,1.19777,1.774473,240.064629,40.692864,64,32.96,1.141000e+10
3,1.141000e+13,1.36989,1.613815,239.498266,39.966295,65,32.96,1.141000e+10
4,1.141000e+13,1.28132,1.545229,206.360391,42.379068,66,35.96,1.141000e+10


In [17]:
results = []

for reach_id, group in average_df.groupby("reach_id"):
    signal = group["wse_median"].values

    infos_list, has_step = detect_step_change(
        signal,
        group["rdr_sig0"].values,
        group["occurrence"].values,
        group["wse_std"].values,
        group["width"].values,
        stability_scale=0.3,
    )

    if has_step:
        for infos in infos_list:
            step_idx = infos[0]

            results.append({
                "reach_id": reach_id,
                "node_id": group["node_id"].iloc[step_idx],
                "step_index": step_idx,
                "change_size": infos[1],
                "change_size1": infos[2],
                "change_size2": infos[3],
                "change_size3": infos[4],
                "change_ratio": infos[1] / infos[5] if infos[5] != 0 else np.nan,
                "sigma": infos[6],
                "sigma2": infos[7],
                "sigma3": infos[8],
                "sigma4": infos[9],
                "occu": infos[10],
                "occu2": infos[11],
                "occu3": infos[12],
                "occu4": infos[13],
                "time_std": infos[14],
                "before_std": infos[15],
                "after_std": infos[16],
                "before_slope": infos[17],
                "after_slope": infos[18],
                "width_mean": infos[19],
                "width_std": infos[20],
            })

step_df = pd.DataFrame(results)
step_df.head()

,reach_id,node_id,step_index,change_size,change_size1,change_size2,change_size3,change_ratio,sigma,sigma2,...,occu2,occu3,occu4,time_std,before_std,after_std,before_slope,after_slope,width_mean,width_std
0,1.144130e+10,1.144130e+13,45,0.713410,0.713410,1.073501,-1.073501,0.140847,56.838035,55.866950,...,26.712,26.78,0.88,0.973388,0.080516,0.138927,0.061826,0.076740,207.239169,44.931627
1,1.144130e+10,1.144130e+13,22,0.484487,0.517915,0.484487,-0.484487,0.190263,39.778581,38.300100,...,23.920,43.15,32.04,0.348343,0.084062,0.068661,0.009883,0.013094,186.490239,18.157586
2,1.144700e+10,1.144700e+13,17,0.885540,0.885540,1.074885,-1.074885,0.120432,51.050180,41.088800,...,8.192,30.34,19.96,0.093907,0.138719,0.101401,0.088781,0.054386,113.507106,44.772644
3,1.144700e+10,1.144700e+13,8,1.128015,1.128015,1.438881,-1.438881,0.189067,23.435853,26.331840,...,18.912,35.78,15.40,0.092274,0.261495,0.117909,0.232202,0.065293,147.371840,27.273738
4,1.144900e+10,1.144900e+13,5,3.030009,3.716000,3.030009,-3.030009,0.067123,29.008159,28.841114,...,2.648,4.38,0.00,0.013304,0.596418,0.258141,1.413934,0.983422,162.739474,49.045056


In [18]:
output_file = OUTPUT_DIR / "train_valid_step_candidates_v6.8.csv"
step_df.to_csv(output_file, index=False)

print(f"Saved detection results to {output_file}")
print(f"Number of detected candidate steps: {len(step_df)}")

Saved detection results to D:\github\SWOT_RiverSP_Obstruction\data\results\train_valid_step_candidates_v6.8.csv
Number of detected candidate steps: 2546


# 2 Prepare CC-based features for annotated candidate nodes


This step derives **profile-correlation features** for the annotated
candidate nodes identified in the pilot experiment.

For each annotated node, a local window centered on the node is extracted along
its reach. The daily WSE profiles within that window are then compared with the
median WSE profile of the same reach using Pearson correlation. The resulting
CC statistics summarize how consistently the local daily profile matches the
reference median profile through time.

The output of this step is a node-level feature table containing
summary statistics of the correlation coefficients and window
characteristics, which will be used for subsequent model training.

These features are merged back into the annotated candidate table for
subsequent analysis and model development.

In [19]:
def _window_corr_matrix(Y, X, left, right):
    """
    Compute window-based Pearson correlations between daily WSE profiles
    and the median WSE profile for all node positions within a reach.

    Parameters
    ----------
    Y : np.ndarray
        Daily WSE matrix with shape (D, N), where D is the number of dates
        and N is the number of nodes.
    X : np.ndarray
        Median WSE profile with shape (N,).
    left : np.ndarray
        Left window bounds for each node position.
    right : np.ndarray
        Right window bounds for each node position (Python slice style).

    Returns
    -------
    tuple
        corr : np.ndarray
            Correlation matrix with shape (D, N).
        nwin : np.ndarray
            Number of valid paired observations for each window, shape (D, N).
    """
    D, N = Y.shape
    X2 = X * X

    # Valid mask: both daily and median values must be finite
    M = np.isfinite(Y) & np.isfinite(X)[None, :]
    Mf = M.astype(float)

    # Replace invalid entries with zero for cumulative sums
    Y0 = np.where(M, Y, 0.0)
    Y20 = np.where(M, Y * Y, 0.0)
    X0 = X[None, :] * Mf
    X20 = X2[None, :] * Mf
    XY0 = X[None, :] * Y0

    def cumsum_pad(A):
        cs = np.cumsum(A, axis=1)
        return np.concatenate([np.zeros((D, 1), dtype=float), cs], axis=1)

    csY = cumsum_pad(Y0)
    csY2 = cumsum_pad(Y20)
    csX = cumsum_pad(X0)
    csX2 = cumsum_pad(X20)
    csXY = cumsum_pad(XY0)
    csN = cumsum_pad(Mf)

    # Window sums for all positions
    sumY = csY[:, right] - csY[:, left]
    sumY2 = csY2[:, right] - csY2[:, left]
    sumX = csX[:, right] - csX[:, left]
    sumX2 = csX2[:, right] - csX2[:, left]
    sumXY = csXY[:, right] - csXY[:, left]
    nwin = csN[:, right] - csN[:, left]

    # Pearson correlation
    num = nwin * sumXY - sumX * sumY
    denx = nwin * sumX2 - sumX * sumX
    deny = nwin * sumY2 - sumY * sumY
    den = np.sqrt(denx * deny)

    corr = np.full((D, N), np.nan, dtype=float)
    ok = (nwin >= 2) & (den > 0)
    corr[ok] = num[ok] / den[ok]

    return corr, nwin

In [20]:
def compute_node_profile_cc_fast(df, df_ref, window=5, ref_node_col=None):
    """
    Compute local profile-correlation features for selected reference nodes.

    Parameters
    ----------
    df : pd.DataFrame
        Input table containing at least:
        ['reach_id', 'node_id', 'date', 'wse'].
        Repeated observations are aggregated internally.
    df_ref : pd.DataFrame
        Reference table containing the target node list.
    window : int, optional
        Half-window size along the reach.
    ref_node_col : str, optional
        Column name in df_ref containing node IDs. Defaults to 'node_id'.

    Returns
    -------
    pd.DataFrame
        Node-level CC feature table.
    """
    df = df.copy()
    df["node_id"] = df["node_id"].astype(int)
    df["date"] = pd.to_datetime(df["date"])

    if ref_node_col is None:
        ref_node_col = "node_id"

    ref_nodes = df_ref[ref_node_col].astype(int).unique()

    # Median WSE profile for each node
    median_prof = (
        df.groupby(["reach_id", "node_id"], as_index=False)["wse"]
        .median()
        .rename(columns={"wse": "wse_median"})
    )

    # Daily WSE profile (median per node per day)
    daily_prof = (
        df.groupby(["reach_id", "date", "node_id"], as_index=False)["wse"]
        .median()
        .rename(columns={"wse": "wse_daily"})
    )

    # Derive reach_id for the reference nodes
    ref_df = pd.DataFrame({"node_id": ref_nodes})
    ref_df["reach_id"] = ref_df["node_id"] // 10000 * 10 + ref_df["node_id"] % 10

    results = []

    # Process one reach at a time
    for reach_id, gref in ref_df.groupby("reach_id"):
        nodes = (
            median_prof.loc[median_prof["reach_id"] == reach_id, "node_id"]
            .astype(int)
            .unique()
        )

        if len(nodes) == 0:
            continue

        nodes = np.sort(nodes)
        N = len(nodes)

        # Median profile
        X = (
            median_prof[median_prof["reach_id"] == reach_id]
            .set_index("node_id")
            .reindex(nodes)["wse_median"]
            .to_numpy(dtype=float)
        )

        # Daily profile matrix
        sub_daily = daily_prof[daily_prof["reach_id"] == reach_id]
        if sub_daily.empty:
            for nid in gref["node_id"].tolist():
                results.append(
                    dict(
                        node_id=nid,
                        reach_id=reach_id,
                        cc_mean=np.nan,
                        cc_median=np.nan,
                        cc_count=0,
                        valid_nodes_mean=np.nan,
                        valid_nodes_median=np.nan,
                        window_size=0,
                    )
                )
            continue

        Ydf = (
            sub_daily.pivot(index="date", columns="node_id", values="wse_daily")
            .reindex(columns=nodes)
        )
        Y = Ydf.to_numpy(dtype=float)

        # Window bounds for all node positions
        pos = np.arange(N)
        left = np.maximum(0, pos - window)
        right = np.minimum(N, pos + window + 1)

        # Compute correlations for all positions
        corr, nwin = _window_corr_matrix(Y, X, left, right)

        node_to_pos = {int(n): i for i, n in enumerate(nodes)}

        for nid in gref["node_id"].tolist():
            if int(nid) not in node_to_pos:
                results.append(
                    dict(
                        node_id=int(nid),
                        reach_id=reach_id,
                        cc_mean=np.nan,
                        cc_median=np.nan,
                        cc_count=0,
                        valid_nodes_mean=np.nan,
                        valid_nodes_median=np.nan,
                        window_size=0,
                    )
                )
                continue

            p = node_to_pos[int(nid)]
            s = corr[:, p]
            nn = nwin[:, p]

            cc_count = int(np.isfinite(s).sum())
            cc_mean = float(np.nanmean(s)) if cc_count else np.nan
            cc_median = float(np.nanmedian(s)) if cc_count else np.nan

            vn_count = int(np.isfinite(nn).sum())
            vn_mean = float(np.nanmean(nn)) if vn_count else np.nan
            vn_median = float(np.nanmedian(nn)) if vn_count else np.nan

            win_size = int(right[p] - left[p])

            results.append(
                dict(
                    node_id=int(nid),
                    reach_id=reach_id,
                    cc_mean=cc_mean,
                    cc_median=cc_median,
                    cc_count=cc_count,
                    valid_nodes_mean=vn_mean,
                    valid_nodes_median=vn_median,
                    window_size=win_size,
                )
            )

    out = (
        pd.DataFrame(results)
        .sort_values(["reach_id", "node_id"])
        .reset_index(drop=True)
    )

    return out

In [21]:
# Input files
annotated_file = OUTPUT_DIR / "train_valid_step_candidates_v6.8_ANNOTATED.csv"
train_file = INPUT_DIR / "node_train_valid_v5.10.csv"

df_ref = pd.read_csv(annotated_file)
df = pd.read_csv(train_file)

# Derive reach_id and calendar date
df["reach_id"] = df["node_id"] // 10000 * 10 + df["node_id"] % 10
df["time_str"] = pd.to_datetime(df["time"], unit="s", origin="2000-01-01")
df["date"] = pd.to_datetime(df["time_str"].dt.date)

df_ref.head()

,Unnamed: 0.1,Unnamed: 0,reach_id,node_id,T/F,N/A,note2,type,step_index,change_size,...,time_std,before_std,after_std,before_slope,after_slope,width_mean,width_std,cc_mean,cc_median,cc_count
0,0,0,11441300371,11441300370461,N,NaN,tributary,NaN,45,0.713410,...,0.973388,0.080516,0.138927,0.061826,0.076740,207.239169,44.931627,0.871084,0.975660,22
1,1,1,11441300421,11441300420231,N,NaN,tributary,NaN,22,0.484487,...,0.348343,0.084062,0.068661,0.009883,0.013094,186.490239,18.157586,0.894157,0.943906,24
2,2,2,11447002081,11447002080181,N,NaN,tributary,NaN,17,0.885540,...,0.093907,0.138719,0.101401,0.088781,0.054386,113.507106,44.772644,0.917188,0.975640,10
3,3,3,11447002141,11447002140091,Y,N,tributary,rapid,8,1.128015,...,0.092274,0.261495,0.117909,0.232202,0.065293,147.371840,27.273738,0.991177,0.995455,20
4,4,4,11449000091,11449000090061,Y,N,NaN,rapid,5,3.030009,...,0.013304,0.596418,0.258141,1.413934,0.983422,162.739474,49.045056,0.999634,0.999785,21


In [23]:
node_cc = compute_node_profile_cc_fast(df, df_ref, window=5)
node_cc.head()

C:\Users\a1106\AppData\Local\Temp\ipykernel_15292\1845613031.py:63: RuntimeWarning: invalid value encountered in sqrt
  den = np.sqrt(denx * deny)
C:\Users\a1106\AppData\Local\Temp\ipykernel_15292\1845613031.py:63: RuntimeWarning: invalid value encountered in sqrt
  den = np.sqrt(denx * deny)
C:\Users\a1106\AppData\Local\Temp\ipykernel_15292\1845613031.py:63: RuntimeWarning: invalid value encountered in sqrt
  den = np.sqrt(denx * deny)
C:\Users\a1106\AppData\Local\Temp\ipykernel_15292\1845613031.py:63: RuntimeWarning: invalid value encountered in sqrt
  den = np.sqrt(denx * deny)
C:\Users\a1106\AppData\Local\Temp\ipykernel_15292\1845613031.py:63: RuntimeWarning: invalid value encountered in sqrt
  den = np.sqrt(denx * deny)
C:\Users\a1106\AppData\Local\Temp\ipykernel_15292\1845613031.py:63: RuntimeWarning: invalid value encountered in sqrt
  den = np.sqrt(denx * deny)
C:\Users\a1106\AppData\Local\Temp\ipykernel_15292\1845613031.py:63: RuntimeWarning: invalid value encountered in sqrt


,node_id,reach_id,cc_mean,cc_median,cc_count,valid_nodes_mean,valid_nodes_median,window_size
0,11441300370461,11441300371,0.871084,0.975660,22,10.863636,11.0,11
1,11441300420231,11441300421,0.894157,0.943906,24,11.000000,11.0,11
2,11447002080181,11447002081,0.917188,0.975640,10,10.900000,11.0,11
3,11447002140091,11447002141,0.991177,0.995455,20,10.800000,11.0,11
4,11449000090061,11449000091,0.999634,0.999785,21,11.000000,11.0,11


In [27]:
df_annotated = pd.read_csv(annotated_file)

df_annotated = df_annotated.merge(
    node_cc[["node_id", "cc_mean", "cc_median", "cc_count"]],
    on="node_id",
    how="left",
)

df_annotated.shape[0]

2546

In [ ]:
output_file = OUTPUT_DIR / "train_valid_step_candidates_v6.8_ANNOTATED.csv"
df_annotated.to_csv(output_file, index=False)

print(f"Saved updated annotated table to {output_file}")

# 3 Search candidate step changes globally and prepare CC-based features


This step extends the candidate search and feature preparation from the annotated `train_valid` subset
to the continent-level `test` datasets at the global scale.

For each continent-specific test dataset, node-level summary statistics are
first computed and merged with ancillary variables including mean sigma0 and
mean GSW occurrence. The step-change detection algorithm is then applied
reach by reach to identify candidate nodes.

After candidate generation, local profile-correlation (CC) features are
computed for all detected candidates using the original daily node
observations. The resulting continent-level candidate tables are merged into a
single global candidate dataset for subsequent prediction and screening.

The outputs of this step include continent-level candidate tables,
their associated CC-based features, and a merged global dataset
containing all detected candidates.

In [51]:
# Ancillary node-level features
sigma_file = INPUT_DIR / "node_sigma0_v5.10.csv"
occu_file = INPUT_DIR / "node_occu_mean.csv"

df_sig = pd.read_csv(sigma_file)
df_occu = pd.read_csv(occu_file)

# Replace nodata occurrence values
df_occu.loc[df_occu["occurrence"] == 255, "occurrence"] = 0

In [52]:
name_list = ["af", "eu", "si", "as", "au", "sa", "na", "ar", "gr"]


for name in name_list:
    input_file = INPUT_DIR / f"node_{name}_test_v5.10.csv"
    df = pd.read_csv(input_file)

    # Derive reach_id
    df["reach_id"] = df["node_id"] // 10000 * 10 + df["node_id"] % 10

    # Compute node-level summary statistics
    median_df = (
        df.groupby("node_id")["wse"]
        .median()
        .reset_index(name="wse_median")
    )

    std_df = (
        df.groupby("node_id")["wse"]
        .std()
        .reset_index(name="wse_std")
    )

    width_df = (
        df.groupby("node_id")["width"]
        .mean()
        .reset_index(name="width")
    )

    # Merge node-level summary statistics
    average_df = median_df.merge(std_df, on="node_id")
    average_df = average_df.merge(width_df, on="node_id")

    # Merge ancillary variables
    average_df = average_df.merge(df_sig, on="node_id")
    average_df = average_df.merge(df_occu, on="node_id")

    # Sort nodes and derive reach_id
    average_df = average_df.sort_values("node_id").reset_index(drop=True)
    average_df["reach_id"] = (
        average_df["node_id"] // 10000 * 10 + average_df["node_id"] % 10
    )

    results = []

    for reach_id, group in average_df.groupby("reach_id"):
        group = group.sort_values("node_id").reset_index(drop=True)
        signal = group["wse_median"].values

        infos_list, has_step = detect_step_change(
            signal,
            group["rdr_sig0"].values,
            group["occurrence"].values,
            group["wse_std"].values,
            group["width"].values,
            stability_scale=0.3,
        )

        if has_step:
            for infos in infos_list:
                step_idx = infos[0]

                results.append({
                    "reach_id": reach_id,
                    "node_id": group["node_id"].iloc[step_idx],
                    "step_index": step_idx,
                    "change_size": infos[1],
                    "change_size1": infos[2],
                    "change_size2": infos[3],
                    "change_size3": infos[4],
                    "change_ratio": infos[1] / infos[5] if infos[5] != 0 else np.nan,
                    "sigma": infos[6],
                    "sigma2": infos[7],
                    "sigma3": infos[8],
                    "sigma4": infos[9],
                    "occu": infos[10],
                    "occu2": infos[11],
                    "occu3": infos[12],
                    "occu4": infos[13],
                    "time_std": infos[14],
                    "before_std": infos[15],
                    "after_std": infos[16],
                    "before_slope": infos[17],
                    "after_slope": infos[18],
                    "width_mean": infos[19],
                    "width_std": infos[20],
                })

    step_df = pd.DataFrame(results)
    output_file = OUTPUT_DIR / f"{name}_test_step_candidates_v6.8.csv"
    step_df.to_csv(output_file, index=False)

    print(f"Saved candidate table for {name} to {output_file}")

Saved candidate table for af to D:\github\SWOT_RiverSP_Obstruction\data\results\af_test_step_candidates_v6.8.csv
Saved candidate table for eu to D:\github\SWOT_RiverSP_Obstruction\data\results\eu_test_step_candidates_v6.8.csv
Saved candidate table for si to D:\github\SWOT_RiverSP_Obstruction\data\results\si_test_step_candidates_v6.8.csv
Saved candidate table for as to D:\github\SWOT_RiverSP_Obstruction\data\results\as_test_step_candidates_v6.8.csv
Saved candidate table for au to D:\github\SWOT_RiverSP_Obstruction\data\results\au_test_step_candidates_v6.8.csv
Saved candidate table for sa to D:\github\SWOT_RiverSP_Obstruction\data\results\sa_test_step_candidates_v6.8.csv
Saved candidate table for na to D:\github\SWOT_RiverSP_Obstruction\data\results\na_test_step_candidates_v6.8.csv
Saved candidate table for ar to D:\github\SWOT_RiverSP_Obstruction\data\results\ar_test_step_candidates_v6.8.csv
Saved candidate table for gr to D:\github\SWOT_RiverSP_Obstruction\data\results\gr_test_step_can

In [53]:
def _window_corr_matrix(Y, X, left, right):
    """
    Compute window-based Pearson correlations between daily WSE profiles
    and the median WSE profile for all node positions within a reach.
    """
    D, N = Y.shape
    X2 = X * X

    M = np.isfinite(Y) & np.isfinite(X)[None, :]
    Mf = M.astype(float)

    Y0 = np.where(M, Y, 0.0)
    Y20 = np.where(M, Y * Y, 0.0)
    X0 = X[None, :] * Mf
    X20 = X2[None, :] * Mf
    XY0 = X[None, :] * Y0

    def cumsum_pad(A):
        cs = np.cumsum(A, axis=1)
        return np.concatenate([np.zeros((D, 1), dtype=float), cs], axis=1)

    csY = cumsum_pad(Y0)
    csY2 = cumsum_pad(Y20)
    csX = cumsum_pad(X0)
    csX2 = cumsum_pad(X20)
    csXY = cumsum_pad(XY0)
    csN = cumsum_pad(Mf)

    sumY = csY[:, right] - csY[:, left]
    sumY2 = csY2[:, right] - csY2[:, left]
    sumX = csX[:, right] - csX[:, left]
    sumX2 = csX2[:, right] - csX2[:, left]
    sumXY = csXY[:, right] - csXY[:, left]
    nwin = csN[:, right] - csN[:, left]

    num = nwin * sumXY - sumX * sumY
    denx = nwin * sumX2 - sumX * sumX
    deny = nwin * sumY2 - sumY * sumY
    den = np.sqrt(denx * deny)

    corr = np.full((D, N), np.nan, dtype=float)
    ok = (nwin >= 2) & (den > 0)
    corr[ok] = num[ok] / den[ok]

    return corr, nwin

In [54]:
def compute_node_profile_cc_fast(df, df_ref, window=5, ref_node_col=None):
    """
    Compute local profile-correlation features for selected candidate nodes.
    """
    df = df.copy()
    df["node_id"] = df["node_id"].astype(int)
    df["date"] = pd.to_datetime(df["date"])

    if ref_node_col is None:
        ref_node_col = "node_id"

    ref_nodes = df_ref[ref_node_col].astype(int).unique()

    median_prof = (
        df.groupby(["reach_id", "node_id"], as_index=False)["wse"]
        .median()
        .rename(columns={"wse": "wse_median"})
    )

    daily_prof = (
        df.groupby(["reach_id", "date", "node_id"], as_index=False)["wse"]
        .median()
        .rename(columns={"wse": "wse_daily"})
    )

    ref_df = pd.DataFrame({"node_id": ref_nodes})
    ref_df["reach_id"] = ref_df["node_id"] // 10000 * 10 + ref_df["node_id"] % 10

    results = []

    for reach_id, gref in ref_df.groupby("reach_id"):
        nodes = (
            median_prof.loc[median_prof["reach_id"] == reach_id, "node_id"]
            .astype(int)
            .unique()
        )

        if len(nodes) == 0:
            continue

        nodes = np.sort(nodes)
        N = len(nodes)

        X = (
            median_prof[median_prof["reach_id"] == reach_id]
            .set_index("node_id")
            .reindex(nodes)["wse_median"]
            .to_numpy(dtype=float)
        )

        sub_daily = daily_prof[daily_prof["reach_id"] == reach_id]
        if sub_daily.empty:
            for nid in gref["node_id"].tolist():
                results.append(
                    dict(
                        node_id=nid,
                        reach_id=reach_id,
                        cc_mean=np.nan,
                        cc_median=np.nan,
                        cc_count=0,
                        valid_nodes_mean=np.nan,
                        valid_nodes_median=np.nan,
                        window_size=0,
                    )
                )
            continue

        Ydf = (
            sub_daily.pivot(index="date", columns="node_id", values="wse_daily")
            .reindex(columns=nodes)
        )
        Y = Ydf.to_numpy(dtype=float)

        pos = np.arange(N)
        left = np.maximum(0, pos - window)
        right = np.minimum(N, pos + window + 1)

        corr, nwin = _window_corr_matrix(Y, X, left, right)

        node_to_pos = {int(n): i for i, n in enumerate(nodes)}

        for nid in gref["node_id"].tolist():
            if int(nid) not in node_to_pos:
                results.append(
                    dict(
                        node_id=int(nid),
                        reach_id=reach_id,
                        cc_mean=np.nan,
                        cc_median=np.nan,
                        cc_count=0,
                        valid_nodes_mean=np.nan,
                        valid_nodes_median=np.nan,
                        window_size=0,
                    )
                )
                continue

            p = node_to_pos[int(nid)]
            s = corr[:, p]
            nn = nwin[:, p]

            cc_count = int(np.isfinite(s).sum())
            cc_mean = float(np.nanmean(s)) if cc_count else np.nan
            cc_median = float(np.nanmedian(s)) if cc_count else np.nan

            vn_count = int(np.isfinite(nn).sum())
            vn_mean = float(np.nanmean(nn)) if vn_count else np.nan
            vn_median = float(np.nanmedian(nn)) if vn_count else np.nan

            win_size = int(right[p] - left[p])

            results.append(
                dict(
                    node_id=int(nid),
                    reach_id=reach_id,
                    cc_mean=cc_mean,
                    cc_median=cc_median,
                    cc_count=cc_count,
                    valid_nodes_mean=vn_mean,
                    valid_nodes_median=vn_median,
                    window_size=win_size,
                )
            )

    return (
        pd.DataFrame(results)
        .sort_values(["reach_id", "node_id"])
        .reset_index(drop=True)
    )

In [56]:
name_list = ["af", "eu", "si", "as", "au", "sa", "na", "ar"]

cc_tables = []

for name in name_list:
    test_file = INPUT_DIR / f"node_{name}_test_v5.10.csv"
    candidate_file = OUTPUT_DIR / f"{name}_test_step_candidates_v6.8.csv"

    if not test_file.exists():
        print(f"Missing test file: {test_file}")
        continue

    if not candidate_file.exists():
        print(f"Missing candidate file: {candidate_file}")
        continue

    df = pd.read_csv(test_file)
    df_ref = pd.read_csv(candidate_file)

    if df_ref.empty:
        print(f"No candidates found for {name}")
        continue

    # Derive reach_id and date
    df["reach_id"] = df["node_id"] // 10000 * 10 + df["node_id"] % 10
    df["time_str"] = pd.to_datetime(df["time"], unit="s", origin="2000-01-01")
    df["date"] = pd.to_datetime(df["time_str"].dt.date)

    node_cc = compute_node_profile_cc_fast(df, df_ref, window=5)

    df_ref = df_ref.merge(
        node_cc[["node_id", "cc_mean", "cc_median", "cc_count"]],
        on="node_id",
        how="left",
    )

    cc_tables.append(df_ref)
    print(f"Finished CC features for {name}")

C:\Users\a1106\AppData\Local\Temp\ipykernel_15292\4113552811.py:39: RuntimeWarning: invalid value encountered in sqrt
  den = np.sqrt(denx * deny)
C:\Users\a1106\AppData\Local\Temp\ipykernel_15292\4113552811.py:39: RuntimeWarning: invalid value encountered in sqrt
  den = np.sqrt(denx * deny)
C:\Users\a1106\AppData\Local\Temp\ipykernel_15292\4113552811.py:39: RuntimeWarning: invalid value encountered in sqrt
  den = np.sqrt(denx * deny)
C:\Users\a1106\AppData\Local\Temp\ipykernel_15292\4113552811.py:39: RuntimeWarning: invalid value encountered in sqrt
  den = np.sqrt(denx * deny)
C:\Users\a1106\AppData\Local\Temp\ipykernel_15292\4113552811.py:39: RuntimeWarning: invalid value encountered in sqrt
  den = np.sqrt(denx * deny)
C:\Users\a1106\AppData\Local\Temp\ipykernel_15292\4113552811.py:39: RuntimeWarning: invalid value encountered in sqrt
  den = np.sqrt(denx * deny)
C:\Users\a1106\AppData\Local\Temp\ipykernel_15292\4113552811.py:39: RuntimeWarning: invalid value encountered in sqrt


Finished CC features for af


C:\Users\a1106\AppData\Local\Temp\ipykernel_15292\4113552811.py:39: RuntimeWarning: invalid value encountered in sqrt
  den = np.sqrt(denx * deny)
C:\Users\a1106\AppData\Local\Temp\ipykernel_15292\4113552811.py:39: RuntimeWarning: invalid value encountered in sqrt
  den = np.sqrt(denx * deny)
C:\Users\a1106\AppData\Local\Temp\ipykernel_15292\4113552811.py:39: RuntimeWarning: invalid value encountered in sqrt
  den = np.sqrt(denx * deny)
C:\Users\a1106\AppData\Local\Temp\ipykernel_15292\4113552811.py:39: RuntimeWarning: invalid value encountered in sqrt
  den = np.sqrt(denx * deny)
C:\Users\a1106\AppData\Local\Temp\ipykernel_15292\4113552811.py:39: RuntimeWarning: invalid value encountered in sqrt
  den = np.sqrt(denx * deny)
C:\Users\a1106\AppData\Local\Temp\ipykernel_15292\4113552811.py:39: RuntimeWarning: invalid value encountered in sqrt
  den = np.sqrt(denx * deny)
C:\Users\a1106\AppData\Local\Temp\ipykernel_15292\4113552811.py:39: RuntimeWarning: invalid value encountered in sqrt


Finished CC features for eu


C:\Users\a1106\AppData\Local\Temp\ipykernel_15292\4113552811.py:39: RuntimeWarning: invalid value encountered in sqrt
  den = np.sqrt(denx * deny)
C:\Users\a1106\AppData\Local\Temp\ipykernel_15292\4113552811.py:39: RuntimeWarning: invalid value encountered in sqrt
  den = np.sqrt(denx * deny)


...

Finished CC features for sa


C:\Users\a1106\AppData\Local\Temp\ipykernel_15292\4113552811.py:39: RuntimeWarning: invalid value encountered in sqrt
  den = np.sqrt(denx * deny)
C:\Users\a1106\AppData\Local\Temp\ipykernel_15292\4113552811.py:39: RuntimeWarning: invalid value encountered in sqrt
  den = np.sqrt(denx * deny)
C:\Users\a1106\AppData\Local\Temp\ipykernel_15292\4113552811.py:39: RuntimeWarning: invalid value encountered in sqrt
  den = np.sqrt(denx * deny)
C:\Users\a1106\AppData\Local\Temp\ipykernel_15292\4113552811.py:39: RuntimeWarning: invalid value encountered in sqrt
  den = np.sqrt(denx * deny)
C:\Users\a1106\AppData\Local\Temp\ipykernel_15292\4113552811.py:39: RuntimeWarning: invalid value encountered in sqrt
  den = np.sqrt(denx * deny)
C:\Users\a1106\AppData\Local\Temp\ipykernel_15292\4113552811.py:39: RuntimeWarning: invalid value encountered in sqrt
  den = np.sqrt(denx * deny)
C:\Users\a1106\AppData\Local\Temp\ipykernel_15292\4113552811.py:39: RuntimeWarning: invalid value encountered in sqrt


Finished CC features for na


C:\Users\a1106\AppData\Local\Temp\ipykernel_15292\4113552811.py:39: RuntimeWarning: invalid value encountered in sqrt
  den = np.sqrt(denx * deny)
C:\Users\a1106\AppData\Local\Temp\ipykernel_15292\4113552811.py:39: RuntimeWarning: invalid value encountered in sqrt
  den = np.sqrt(denx * deny)
C:\Users\a1106\AppData\Local\Temp\ipykernel_15292\4113552811.py:39: RuntimeWarning: invalid value encountered in sqrt
  den = np.sqrt(denx * deny)


Finished CC features for ar


In [57]:
if not cc_tables:
    raise ValueError("No global candidate tables were generated.")

df_global = pd.concat(cc_tables, ignore_index=True)

global_output_file = OUTPUT_DIR / "test_step_candidates_v6.8.csv"
df_global.to_csv(global_output_file, index=False)

print(f"Saved global candidate table to {global_output_file}")
print(f"Number of global candidates: {len(df_global)}")

Saved global candidate table to D:\github\SWOT_RiverSP_Obstruction\data\results\test_step_candidates_v6.8.csv
Number of global candidates: 22951


# 4 Train and apply a Random Forest classifier on the annotated dataset


This step trains a Random Forest classifier on the full annotated pilot dataset
using the fixed best hyperparameters selected earlier in `Experiment-15.ipynb`.

There is no train/validation split, no model comparison, and no hold-out result display here.
The goal of this section is only to:

1. train the final RF model on all annotated pilot samples;
2. apply that trained model to the global candidate dataset;
3. save predicted scores and the predicted positive subset.


In [5]:
from sklearn.ensemble import RandomForestClassifier

# Best RF parameters copied from Experiment-15.ipynb
BEST_RF_PARAMS = {
    "n_estimators": 500,
    "max_depth": 8,
    "min_samples_leaf": 10,
    "max_features": "sqrt",
}

PRED_THRESHOLD = 0.5

BEST_RF_PARAMS


{'n_estimators': 500,
 'max_depth': 8,
 'min_samples_leaf': 10,
 'max_features': 'sqrt'}

In [6]:
# Input annotated pilot data
annotated_file = OUTPUT_DIR / "train_valid_step_candidates_v6.8_ANNOTATED.csv"
df = pd.read_csv(annotated_file)

# Keep only samples with manual labels
df = df.dropna(subset=["T/F"]).copy()

# Convert labels to binary format
df["label"] = (df["T/F"] == "Y").astype(int)

print("Annotated pilot samples:", len(df))
print("Positive samples:", int(df["label"].sum()))
print("Negative samples:", int((1 - df["label"]).sum()))

df.head()


Annotated pilot samples: 2546
Positive samples: 734
Negative samples: 1812


,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,reach_id,node_id,T/F,N/A,note2,type,step_index,...,before_std,after_std,before_slope,after_slope,width_mean,width_std,cc_mean,cc_median,cc_count,label
0,0,0,0,11441300371,11441300370461,N,NaN,tributary,NaN,45,...,0.080516,0.138927,0.061826,0.076740,207.239169,44.931627,0.871084,0.975660,22,0
1,1,1,1,11441300421,11441300420231,N,NaN,tributary,NaN,22,...,0.084062,0.068661,0.009883,0.013094,186.490239,18.157586,0.894157,0.943906,24,0
2,2,2,2,11447002081,11447002080181,N,NaN,tributary,NaN,17,...,0.138719,0.101401,0.088781,0.054386,113.507106,44.772644,0.917188,0.975640,10,0
3,3,3,3,11447002141,11447002140091,Y,N,tributary,rapid,8,...,0.261495,0.117909,0.232202,0.065293,147.371840,27.273738,0.991177,0.995455,20,1
4,4,4,4,11449000091,11449000090061,Y,N,NaN,rapid,5,...,0.596418,0.258141,1.413934,0.983422,162.739474,49.045056,0.999634,0.999785,21,1


In [7]:
# Selected feature columns
feat_cols = [
    "change_size1",
    "change_size2",
    "change_size3",
    "change_size",
    "sigma",
    "sigma2",
    "sigma3",
    "sigma4",
    "occu",
    "occu2",
    "occu3",
    "occu4",
    "time_std",
    "before_std",
    "after_std",
    "before_slope",
    "after_slope",
    "width_mean",
    "cc_mean",
]

X = df[feat_cols].copy()
X = pd.get_dummies(X, dummy_na=True)
y = df["label"].values

print("Feature matrix shape:", X.shape)
print("Feature columns used:")
print(feat_cols)


Feature matrix shape: (2546, 19)
Feature columns used:
['change_size1', 'change_size2', 'change_size3', 'change_size', 'sigma', 'sigma2', 'sigma3', 'sigma4', 'occu', 'occu2', 'occu3', 'occu4', 'time_std', 'before_std', 'after_std', 'before_slope', 'after_slope', 'width_mean', 'cc_mean']


In [8]:
# Train final RF on the full pilot dataset
rf = RandomForestClassifier(
    random_state=42,
    n_jobs=-1,
    class_weight="balanced",
    **BEST_RF_PARAMS,
)

rf.fit(X, y)

print("Final RF trained on full pilot data.")
print("Training rows:", X.shape[0])
print("Training columns:", X.shape[1])


Final RF trained on full pilot data.
Training rows: 2546
Training columns: 19


In [9]:
# Load the global candidate dataset
global_file = OUTPUT_DIR / "test_step_candidates_v6.8.csv"
df_global = pd.read_csv(global_file)

print("Total global candidates:", len(df_global))

# Prepare the global feature matrix using the same training features
X_global = df_global[feat_cols].copy()
X_global = pd.get_dummies(X_global, dummy_na=True)
X_global = X_global.reindex(columns=X.columns, fill_value=0)

print("Global feature matrix shape:", X_global.shape)


Total global candidates: 22951
Global feature matrix shape: (22951, 19)


In [10]:
# Predict probability + binary label
y_prob = rf.predict_proba(X_global)[:, 1]
y_pred = (y_prob >= PRED_THRESHOLD).astype(int)
y_label = np.where(y_pred == 1, "Y", "N")

df_global_pred = df_global.copy()
df_global_pred["pred_prob"] = y_prob
df_global_pred["pred"] = y_pred
df_global_pred["pred_label"] = y_label

print("Predicted positive candidates:", int((df_global_pred["pred"] == 1).sum()))
print("Predicted positive ratio:", float((df_global_pred["pred"] == 1).mean()))
df_global_pred.head()


Predicted positive candidates: 7781
Predicted positive ratio: 0.3390266219336848


,reach_id,node_id,step_index,change_size,change_size1,change_size2,change_size3,change_ratio,sigma,sigma2,...,before_slope,after_slope,width_mean,width_std,cc_mean,cc_median,cc_count,pred_prob,pred,pred_label
0,1.143000e+10,1.143000e+13,34,0.691510,0.830885,0.691510,-0.691510,0.425142,48.615315,49.658338,...,-0.041781,0.030047,249.219793,124.677981,0.531351,0.576699,33,0.063234,0,N
1,1.144110e+10,1.144110e+13,31,1.052130,1.052130,1.220487,1.220487,0.308860,96.636529,85.545624,...,0.166310,0.044961,303.718949,108.062469,-0.179797,-0.495974,17,0.028676,0,N
2,1.144110e+10,1.144110e+13,45,1.210410,1.210410,1.367759,1.367759,0.355324,102.901621,86.521576,...,0.172060,0.039865,274.765368,93.026636,0.307423,0.457817,17,0.048837,0,N
3,1.144110e+10,1.144110e+13,28,0.558672,0.819080,0.558672,-0.558672,0.166478,143.668365,130.961632,...,-0.003058,-0.020747,174.423526,82.601559,0.333429,0.352935,31,0.096356,0,N
4,1.144110e+10,1.144110e+13,14,0.936139,1.080500,0.936139,0.936139,0.434031,94.330508,84.101436,...,0.025998,0.116685,430.500450,193.149506,0.021626,-0.207400,30,0.020110,0,N


In [11]:
# Save outputs
all_pred_file = OUTPUT_DIR / "global_predicted_v6.8_all.csv"
positive_file = OUTPUT_DIR / "global_predicted_v6.8.csv"

df_global_pred.to_csv(all_pred_file, index=False)

df_positive = df_global_pred[df_global_pred["pred"] == 1].copy()
df_positive.to_csv(positive_file, index=False)

print(f"Saved all predictions to {all_pred_file}")
print(f"Saved predicted positive candidates to {positive_file}")
print("Positive subset rows:", len(df_positive))


Saved all predictions to D:\github\SWOT_RiverSP_Obstruction\data\results\global_predicted_v6.8_all.csv
Saved predicted positive candidates to D:\github\SWOT_RiverSP_Obstruction\data\results\global_predicted_v6.8.csv
Positive subset rows: 7781
